The statistics of the small matrix in *KuaiRec*.

|                | #Users | #Items | #Interactions | Density |
| -------------- | :----: | :----: | :-----------: | :-----: |
| *small matrix* | 1,411  | 3,327  |   4,676,570   |  99.6%  |
| *big matrix*   | 7,176  | 10,728 |  12,530,806   |  16.3%  |

Note that the density of the small matrix is 99.6% instead of 100% because some users have explicitly indicated that they would not be willing to receive recommendations from certain authors. I.e., They blocked these videos.

#### 1. Descriptions of the fields in `small_matrix.csv`. 

| Field Name:    | Description                                              | Type    | Example                   |
| -------------- | -------------------------------------------------------- | ------- | ------------------------- |
| uid        | The ID of the user.                                      | int64   | 0                         |
| video_id       | The ID of the viewed video.                              | int64   | 3650                      |
| play_duration  | Time of video viewing of this interaction (millisecond). | int64   | 13838                     |
| video_duration | Time of this video (millisecond).                        | int64   | 10867                     |
| time           | Human-readable date for this interaction                 | str     | "2020-07-05 00:08:23.438" |
| date           | Date of this interaction                                 | int64   | 20200705                  |
| timestamp      | Unix timestamp                                           | float64 | 1593878903.438            |
| watch_ratio    | The video watching ratio (=play_duration/video_duration) | float64 | 1.273397                  |

The "watch_ratio" can be deemed as the label of the interaction. Note: there is no "like" signal for this dataset. If you need this binary signal in your scenarios, you can create it yourself. E.g., `like = 1 if watch_ratio > 2.0`.

#### 2. Descriptions of the caption and category fields in `kuairec_caption_category.csv` (Added on 2024.06.02)


| Field Name:                | Description                                            | Type  | Example                                                      |
| -------------------------- | ------------------------------------------------------ | ----- | ------------------------------------------------------------ |
| video_id                   | The ID of the video                                    | int64 | 2418                                                         |
| manual_cover_text          | 封面文字 (added by its author)                         | str   | "被小可爱发现了"                                             |
| caption                    | 简介标题 (added by its author)                         | str   | "这是什么狗狗，这么可爱真的可以这么遛吗？#喜欢的双击加关注 #直播 #博美俊介 #萌宠驾到" |
| topic_tag                  | Tags of the topics of this video (added by its author) | str   | "[博美俊介,喜欢的双击加关注,直播,萌宠驾到]"                  |
| first_level_category_id    | First-level category ID                                | int64 | 17                                                           |
| first_level_category_name  | First-level category name                              | str   | "宠物"                                                       |
| second_level_category_id   | Second-level category ID                               | int64 | 233                                                          |
| second_level_category_name | Second-level category name                             | str   | "宠物日常记录"                                               |
| third_level_category_id    | Thrid-level category ID                                | int64 | 1169                                                         |
| third_level_category_name  | Third-level category name                              | str   | "宠物狗"                                                     |

In [1]:
# Loading dataset

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print("Loading...")
small_matrix = pd.read_csv("./data/small_matrix.csv")

big_matrix = pd.read_csv("./data/big_matrix.csv")

kuairec_caption_category = pd.read_csv('./data/kuairec_caption_category.csv', engine='python')
kuairec_caption_category['video_id'] = pd.to_numeric(kuairec_caption_category['video_id'], errors='coerce')
kuairec_caption_category = kuairec_caption_category.dropna(subset=['video_id']).reset_index(drop=True)


Loading...


In [2]:
# Caption generation

def get_valid_text(val):
    if pd.isna(val) or val is None:
        return ""
    
    s_val = str(val).strip()
    
    if s_val == "UNKNOWN":
        return ""
        
    return s_val

def process_row(row):
    cats = [
        get_valid_text(row.get('first_level_category_name')),
        get_valid_text(row.get('second_level_category_name')),
        get_valid_text(row.get('third_level_category_name'))
    ]
    valid_cats = [c for c in cats if c]
    
    cat_str = ""
    if valid_cats:
        cat_str = f"【{' > '.join(valid_cats)}】"

    cover = get_valid_text(row.get('manual_cover_text'))
    caption = get_valid_text(row.get('caption'))
    
    content_parts = []
    if cover: content_parts.append(cover)
    if caption: content_parts.append(caption)
    content_str = " ".join(content_parts)

    raw_tags = str(row.get('topic_tag', '')).strip()
    tag_str = ""
    
    if raw_tags and raw_tags != "[]" and raw_tags.lower() != "nan":
        clean_tags = raw_tags.replace('[', '').replace(']', '').replace('"', '').replace("'", "")
        if clean_tags.strip():
            tags_list = [t.strip() for t in clean_tags.split(',') if t.strip()]
            tag_str = " ".join([f"#{t}" for t in tags_list])

    final_parts = [cat_str, content_str, tag_str]
    final_text = " ".join([p for p in final_parts if p])
    
    return final_text

kuairec_caption_category['text_input'] = kuairec_caption_category.apply(process_row, axis=1)


In [3]:
description_thold = 20
user_drop_ratio = 0.5

# Drop videos with empty description

mask = (kuairec_caption_category['text_input'].str.len() <= description_thold)
kuairec_caption_category_cleaned = kuairec_caption_category[~mask].reset_index(drop=True)
dropped_video_ids = kuairec_caption_category[mask]['video_id'].to_list()
print(f"Dropped {len(dropped_video_ids)} items with empty description.")

# Drop users

big_users = set(big_matrix['user_id'].unique())
small_users = set(small_matrix['user_id'].unique())
only_in_big_users = list(big_users - small_users)
num_to_drop = int(len(only_in_big_users) * user_drop_ratio)

import random
random.seed(42)
users_to_drop = random.sample(only_in_big_users, num_to_drop)

print(f"Dropped {len(users_to_drop)} users in big matrix.")

big_matrix_cleaned = big_matrix[~big_matrix['video_id'].isin(dropped_video_ids)].reset_index(drop=True)
big_matrix_cleaned = big_matrix_cleaned[~big_matrix_cleaned['user_id'].isin(users_to_drop)].reset_index(drop=True)
small_matrix_cleaned = small_matrix[~small_matrix['video_id'].isin(dropped_video_ids)].reset_index(drop=True)

print(f"big_matrix: dropped {len(big_matrix) - len(big_matrix_cleaned)} interactions.")
print(f"small_matrix: dropped {len(small_matrix) - len(small_matrix_cleaned)} interactions, {len(set(small_matrix['video_id']).intersection(set(dropped_video_ids)))} items.")

# id mapping

video_id_mapping = {video_id: idx for idx, video_id in enumerate(kuairec_caption_category_cleaned['video_id'].unique())}
big_matrix_cleaned['vid'] = big_matrix_cleaned['video_id'].map(video_id_mapping)
small_matrix_cleaned['vid'] = small_matrix_cleaned['video_id'].map(video_id_mapping)
kuairec_caption_category_cleaned['vid'] = kuairec_caption_category_cleaned['video_id'].map(video_id_mapping)

user_id_mapping = {user_id: idx for idx, user_id in enumerate(big_matrix_cleaned['user_id'].unique())}
big_matrix_cleaned['uid'] = big_matrix_cleaned['user_id'].map(user_id_mapping)
small_matrix_cleaned['uid'] = small_matrix_cleaned['user_id'].map(user_id_mapping)

# Save caption file

result = kuairec_caption_category_cleaned[['vid', 'text_input']]
print("\nNumber of items with captions: ", len(result))
print("Sample captions:")
result.to_csv('./rec_list/item_caption.csv', index=False)
result.head(5)

Dropped 1987 items with empty description.
Dropped 2882 users in big matrix.
big_matrix: dropped 6653273 interactions.
small_matrix: dropped 417993 interactions, 297 items.

Number of items with captions:  8741
Sample captions:


,vid,text_input
0,0,【颜值 > 颜值随拍】 精神小伙路难走 程哥你狗粮慢点撒
1,1,【喜剧 > 搞笑互动】 晚饭后，运动一下！
2,2,【摄影 > 主题摄影 > 景物摄影】 我平淡无奇，惊艳不了时光，温柔不了岁月，我只想漫无目的...
3,3,【时尚 > 营销售卖 > 女装】 五爱街最美美女 一天1q #搞笑 #感谢快手我要上热门 #...
4,4,【明星娱乐 > 娱乐八卦 > 饭制】 “你们吵的越狠 他们的手就握的越紧” #文轩 ...


In [4]:
# Save big matrix uid-vid list

def build_uid_vid_list(data):
    grouped = data.groupby('uid')['vid'].apply(lambda seq: list(dict.fromkeys(seq))).reset_index()

    # Convert to list-of-lists: [uid, vid1, vid2, ...]
    uid_vid_list = grouped.apply(lambda r: [int(r['uid'])] + [int(v) for v in r['vid']], axis=1).tolist()
    print(f"\nBuilt uid_vid_list for {len(uid_vid_list)} uids. Showing first 5 users with first 20 interactions:")
    
    for row in uid_vid_list[:5]:
        print(row[:20])
        
    return uid_vid_list

big = build_uid_vid_list(big_matrix_cleaned)
with open('./cleaned_data/big_matrix.txt', 'w', encoding='utf-8') as f:
    for row in big:
        f.write(' '.join(map(str, row)) + '\n')
print("Big matrix saved to ./cleaned_data/big_matrix.txt")


Built uid_vid_list for 4294 uids. Showing first 5 users with first 20 interactions:
[0, 2992, 7825, 4319, 1608, 6741, 6736, 5559, 5580, 148, 137, 1629, 4328, 145, 2990, 6748, 167, 5570, 138, 3011]
[1, 4329, 1588, 2992, 148, 4319, 2940, 1610, 7823, 7803, 1621, 116, 4333, 4281, 4327, 2991, 1625, 5559, 4343, 4237]
[2, 7778, 4328, 4288, 5581, 4341, 5580, 7800, 4327, 4321, 3004, 5586, 6724, 138, 1645, 5508, 173, 170, 5558, 3021]
[3, 1608, 1583, 2992, 89, 4333, 2990, 5525, 4288, 2940, 4341, 4319, 7830, 144, 5570, 2967, 7849, 6730, 5537, 1547]
[4, 134, 19, 1586, 5572, 5545, 1598, 7755, 142, 2914, 5558, 1536, 7778, 4318, 116, 122, 4319, 2979, 1583, 1608]
Big matrix saved to ./cleaned_data/big_matrix.txt


In [5]:
# Small Matrix operation: Only keep positive samples: the user has watched the video completely at least twice. 
ratio = 2
data = small_matrix_cleaned[small_matrix_cleaned['watch_ratio'] >= ratio].reset_index(drop=True)

print(f"Number of rows with 'watch_ratio' >= {ratio}: {data.shape[0]}")
print(f"Ratio of rows with 'watch_ratio' >= {ratio}: {data.shape[0] / small_matrix_cleaned.shape[0] }")
print(f"Number of items in filtered small matrix: {data['video_id'].nunique()}")

Number of rows with 'watch_ratio' >= 2: 196536
Ratio of rows with 'watch_ratio' >= 2: 0.04615062731048423
Number of items in filtered small matrix: 2982


In [6]:
# Save small matrix uid-vid list

small = build_uid_vid_list(data)
with open('./cleaned_data/small_matrix.txt', 'w', encoding='utf-8') as f:
    for row in small:
        f.write(' '.join(map(str, row)) + '\n')
print("Small matrix saved to ./cleaned_data/small_matrix.txt")



Built uid_vid_list for 1411 uids. Showing first 5 users with first 20 interactions:
[8, 2992, 5557, 1598, 167, 165, 4370, 6736, 4447, 1738, 1748, 5576, 288, 239, 2944, 7903, 2974, 1812, 3004, 5742]
[11, 252, 122, 6831, 4483, 1742, 4534, 4520, 6962, 5848, 3286, 1916, 3317, 8015, 5557, 3365, 2192, 3392, 4878, 2168]
[13, 122, 7818, 4291, 1629, 6748, 1562, 108, 1662, 134, 4370, 7801, 5613, 4309, 4413, 1646, 6744, 148, 215, 5570]
[15, 1608, 148, 7823, 134, 2992, 4322, 5553, 5557, 3021, 1629, 170, 7877, 4392, 4318, 1671, 3049, 7894, 7904, 4381]
[16, 4317, 167, 202, 5649, 6790, 2969, 1735, 6782, 7821, 6818, 101, 1664, 4463, 303, 246, 7908, 6736, 4466, 4311]
Small matrix saved to ./cleaned_data/small_matrix.txt


In [7]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# import warnings; warnings.simplefilter('ignore')
# def visual_continue(df, func=None):
#     ax = sns.distplot(df)
#     if func:
#         func(ax)
    
#     gca = plt.gca()
#     fig_title = "Statistics of {}".format(df.name)
#     gca.set_title(fig_title, fontsize=14)
#     gca.set_ylabel("Density", fontsize=14)
#     gca.set_xlabel(df.name, fontsize=14)
    
#     plt.show()
    
# small_video_duration = small_matrix.video_duration
# print(small_video_duration.describe())
# # visual_continue(small_video_duration)
# visual_continue(small_video_duration[small_video_duration < 100000])

# small_video_watch = small_matrix.play_duration
# print(small_video_watch.describe())
# # visual_continue(small_video_watch)
# visual_continue(small_video_watch[small_video_watch < 100000])

In [8]:
# Aggregate data based on 'uid' and count how many rows each 'uid' owns
# uid_counts = big_matrix_cleaned.groupby('user_id').size().reset_index(name='row_count')
# print("Aggregated data showing how many rows each 'uid' owns:")
# print(uid_counts.describe()['row_count'])

# def count_uids_below(threshold):
#     return int((uid_counts['row_count'] < threshold).sum())

# for t in [20]:
#     print(f"\nUids with row_count < {t}: {count_uids_below(t)}")

In [9]:
# # Find uids (here column 'user_id') whose row_count < 20 and drop their rows from `data`
# low_uids = uid_counts.loc[uid_counts['row_count'] < 20, 'user_id'].tolist()
# print(f"Number of uids with row_count < 20: {len(low_uids)}")

# # How many rows in `data` belong to these uids
# rows_to_drop = data['user_id'].isin(low_uids).sum()
# print(f"Number of rows to drop from data: {rows_to_drop} out of {data.shape[0]}")

# # Drop the rows and reset index
# data = data[~data['user_id'].isin(low_uids)].reset_index(drop=True)
# print(f"New data shape after dropping: {data.shape}")

# # Update uid_counts to reflect remaining users
# uid_counts = uid_counts[~uid_counts['user_id'].isin(low_uids)].reset_index(drop=True)
# print(f"Remaining unique uids: {uid_counts.shape[0]}")

# data